# LFM2.5-1.2B-Instruct-Uncensored -> Hindi + Hinglish (Kaggle T4, 30-40 min/session)Trains in **chained sessions** so a Kaggle VM reset never costs more than 5 minutes ofwork: each session trains on the *previous* session's pushed output.| session | data file | trains on ||---|---|---|| 1 | `train_s1_hinglish.jsonl` | the original uncensored base || 2 | `train_s2_hindi.jsonl` | session 1's output || 3 | `train_s3_english.jsonl` | session 2's output |Set `SESSION = 1` below, run, then `SESSION = 2`, etc.**Hindi is Devanagari at 1.45 tok/char** (byte-level BPE, no dedicated Devanagari tokens)vs 0.46 for Hinglish. That's why `MAX_SEQ_LENGTH` is tighter here - it is not a bug.**Never finetune a GGUF.** We train the safetensors base and convert back at the end.

In [ ]:
# ---------------------------------------------------------------- run configSESSION   = 1                      # 1, 2 or 3DATA_FILE = {1: "train_s1_hinglish.jsonl",             2: "train_s2_hindi.jsonl",             3: "train_s3_english.jsonl"}[SESSION]HF_USER   = "kumarsujitroy"         # your HF usernameOUT_REPO  = f"{HF_USER}/lfm25-1.2b-bilingual-s{SESSION}"# Session 1 starts from the uncensored base; sessions 2+ continue from the previous push.BASE_REPO = ("zaakirio/LFM2.5-1.2B-Instruct-Uncensored" if SESSION == 1             else f"{HF_USER}/lfm25-1.2b-bilingual-s{SESSION-1}")CHECKPOINT_EVERY_S = 300            # 5 minutes# Session 1 measured 0.893 it/s at 41-token median. Session 2's Hindi rows are# 372-token median / 685 p90 - roughly 6-9x the work per step, so expect ~0.12 it/s# and ~100 min for a full epoch. Lower MAX_STEPS if the session is short.MAX_STEPS = {1: 690, 2: 400, 3: 620}[SESSION]# Chained merged fine-tuning drifts. Each session is a smaller correction on top of# the last, so the LR steps down: s2 is a real new capability, s3 is only an anchor.LEARNING_RATE = {1: 2e-4, 2: 1.5e-4, 3: 5e-5}[SESSION]# Catch a half-edited username before it turns into a confusing 404 ten minutes in.if HF_USER != "kumarsujitroy":    raise ValueError(f"HF_USER looks wrong: {HF_USER!r}")assert SESSION in (1, 2, 3), SESSIONprint(f"session {SESSION}: {DATA_FILE}")print(f"  base  {BASE_REPO}")print(f"  out   {OUT_REPO}")

## 0. Hugging Face login - REQUIRED`push_to_hub` is the whole point: it is the only thing that survives a Kaggle VM reset.**Without a token every checkpoint push fails and you lose the run.**This notebook is **public**, so there is no paste slot for a token anywhere in it -anything typed into a public notebook is published. The token must arrive at runtime.**Setup, once, three steps:**1. huggingface.co -> **Settings -> Access Tokens -> Create new token** -> grant **`write`**2. Right sidebar -> **Add-Ons** tab -> **Secrets** -> **+ New Secret**   - Label `HF_TOKEN`, Value your `hf_...` token, **Save**3. Click **Code Snippet** next to the secret -> **Copy to clipboard**Cell 4 calls `kaggle_secrets.UserSecretsClient().get_secret("HF_TOKEN")`, which is theline Kaggle gives you in that snippet. No Notebook Access toggle, no session restart, norestart dance - just works. It falls back to the `HF_TOKEN` env var if you prefer secrets,and prints `logged in as: kumarsujitroy` when it succeeded.

### Where the model goesThree repos are created automatically on first push - you don't create them by hand:```https://huggingface.co/<you>/lfm25-1.2b-bilingual-s1     <- session 1 outputhttps://huggingface.co/<you>/lfm25-1.2b-bilingual-s2     <- session 2 (loads s1)https://huggingface.co/<you>/lfm25-1.2b-bilingual-s3     <- session 3 (loads s2)```Each holds the merged model, the `*Q4_K_M.gguf`, and a `ckpt-*` adapter pushed every5 min. Repos are **public by default** - the base model is public, so nothing private isleaked, but check the base repo's licence if you plan to redistribute.Back on the server, pull the final GGUF:```hf download <you>/lfm25-1.2b-bilingual-s3 --include "*Q4_K_M*" --local-dir ~/models/@models```**Set `HF_USER` in the run-config cell above first** - it is the only thing you must edit.

In [ ]:
import osfrom huggingface_hub import login, whoami# Two ways in, tried in order. This notebook is PUBLIC, so there is deliberately no# paste slot for a token anywhere in it - anything typed into a public notebook gets# published, and the token must arrive at runtime instead.##  1. kaggle_secrets  <- Kaggle's own mechanism. Right sidebar -> Add-Ons ->#     Secrets -> your HF_TOKEN -> "Code Snippet" -> Copy to clipboard. This is the#     supported path and does not depend on session restart.#  2. os.environ     <- works when HF_TOKEN is injected as a Notebook-Access secret#     at session start.## Setup for (1): Add-Ons -> Secrets -> + New Secret, label HF_TOKEN, paste the# hf_ write token. Nothing to tick, nothing to restart.TOKEN = ""try:    from kaggle_secrets import UserSecretsClient    TOKEN = UserSecretsClient().get_secret("HF_TOKEN") or ""except Exception:    passTOKEN = (TOKEN or os.environ.get("HF_TOKEN", "")).strip()if not TOKEN:    raise RuntimeError(        "No HF token found by either route.\n"        "  Add-Ons (right sidebar) -> Secrets -> + New Secret\n"        "    Label: HF_TOKEN   Value: your hf_ write token\n"        "  Then click the \"Code Snippet\" button next to the secret and Copy to "        "clipboard - that is the exact line used above."    )if not TOKEN.startswith("hf_"):    raise ValueError(f"Token does not look like an HF token: {TOKEN[:6]}...")login(token=TOKEN, add_to_git_credential=False)os.environ["HF_TOKEN"] = TOKENinfo = whoami(token=TOKEN)print("logged in as:", info.get("name"), "| type:", info.get("type"))print("will push to:", f"{info.get('name')}/lfm25-1.2b-bilingual-s{SESSION}")

In [ ]:
!pip install -q unsloth!pip install -q transformers==4.57.3!pip install -q --no-deps trl==0.22.2!pip install -q datasets==4.3.0 accelerate peft bitsandbytes hf_transfer# Import unsloth BEFORE transformers anywhere in this notebook, or it self-warns# and skips its patches. Every later `from transformers import ...` is then safe.import unslothfrom unsloth import FastLanguageModelprint('unsloth imported first - OK')

## Crash-proof checkpointing**A local save on Kaggle is not a backup** - `/kaggle/working` is destroyed when the VMresets, taking every `save_steps` checkpoint with it. The only thing that survives is aHub push, so this callback pushes the LoRA adapter (small, ~60 MB) on a wall-clock timerrather than a step count. If the session dies at minute 38 you resume from minute 35.

In [ ]:
import os, time, json, globfrom transformers import TrainerCallbackclass TimeCheckpoint(TrainerCallback):    """Push the adapter to the Hub every `every_s` seconds of wall clock."""    def __init__(self, model, repo, every_s=300, tokenizer=None):        self.model, self.repo, self.every = model, repo, every_s        self.tok = tokenizer        self.last, self.n = time.time(), 0    def on_step_end(self, args, state, control, **kw):        if time.time() - self.last >= self.every:            self.n += 1            tag = f"ckpt-s{state.global_step//100*100:05d}"            self.model.save_pretrained(f"hf-out/{tag}")            self.model.push_to_hub(self.repo, save_name=tag)            if self.tok is not None:       # adapter alone is not reloadable                self.tok.save_pretrained(f"hf-out/{tag}")                # ...and it has to be on the Hub too, or the checkpoint is only                # half-recoverable. save_pretrained alone keeps it on disk.                self.tok.push_to_hub(self.repo, save_name=tag)            print(f"[{time.strftime('%H:%M:%S')}] pushed {tag} "                  f"@ step {state.global_step}", flush=True)            self.last = time.time()        return controlprint("TimeCheckpoint ready")

## 1. DataUpload the session's jsonl to Kaggle as a Dataset first (Add Data > Upload).

## 1. DataThe cell below needs no manual setup: it uses a Kaggle Dataset if one is attached, andotherwise downloads the file straight from the public GitHub repo(`SujitRoy/hindi-finetune`). Verified reachable: 1.1 MB / 5.0 MB / 2.2 MB.

## 1a. Build the training data (runs here, on Kaggle)Run once; writes `train_s1_hinglish` / `s2_hindi` / `s3_english` into `/kaggle/working/`.Set `BUILD_DATA = False` to skip and use the repo's copies.**Each bucket comes from a different source, because no single dataset is good at allthree.** Measured on the real files:| bucket | source | rows | median reply ||---|---|---|---|| s1 Hinglish | `Sujalvc/hinglish-instruct-dataset` (AAG_anchored) | 9,999 | **68 words** || | `one-thing/chatbot_arena_conversations_hinglish` | real users | — || | `Abhishekcr448/Hinglish-Everyday-Conversations-1M` | natural register | — || s2 Hindi | `smangrul/hindi_instruct_v1` Devanagari | 6,248 | conversational || | `zicsx/indic-align-hindi` (filler) | 512k pool | — || s3 English | `smangrul` mixed-script rows | 4,502 | — |**Why not one dataset for all three.** The first attempt used `smangrul` for everythingand split it by *script*. That failed: its 9,465 romanized rows turned out to be**100% the "Transliteration and Code Mixing" category — translation tasks with amedian 8-word reply.** Training on it produced a model that answered `kya haal hai?`with `Kya har baahar hai`. Validation loss *improved* (2.33 → 1.40) because translationis formulaic, so loss across different data distributions is not comparable and told usnothing about conversational ability.**The check that catches this is median reply words, printed per bucket at the end.**Splitting on script is not enough; the cell now asserts the output is actuallyconversational.Contamination: Marathi and Nepali are also Devanagari, so a script test alone admitsthem. This cell tests Marathi **standalone words** with script-boundary lookarounds andrejects Bengali/Gujarati/Tamil/Telugu/Kannada/Malayalam/Arabic outright.

In [ ]:
# Self-contained data build on Kaggle.BUILD_DATA = True# Every source that measured well gets used. Source count is not a constraint -# each session trains independently and the downloads stream.#   s1  Sujalvc + arena + casual + COMI-LINGUA   (romanized Hinglish)#   s2  smangrul + zicsx + COMI-LINGUA           (Devanagari Hindi)#   s3  smangrul English-bearing rowsHINDI_TARGET    = 6000HINGLISH_TARGET = 8500ENGLISH_TARGET  = 4000if not BUILD_DATA:    print("BUILD_DATA=False - the next cell will use the repo's jsonl")else:    import csv, io, json, os, random, re, urllib.request    def grab(url, dest):        os.makedirs(os.path.dirname(dest), exist_ok=True)        if not os.path.exists(dest):            with urllib.request.urlopen(url, timeout=900) as r, open(dest, "wb") as f:                f.write(r.read())        return dest    def stream_csv(url, dest, i_c, o_c, cap):        """Arena/casual are big remote CSVs - stream them, never hold 1M rows in RAM."""        p = grab(url, dest)        rows, hdr = [], None        with open(p, encoding="utf-8", errors="replace") as f:            for line in io.TextIOWrapper(io.BufferedReader(open(p, "rb"))):                if hdr is None:                    hdr = next(csv.reader([line.decode("utf-8", "replace")])); continue                rec = dict(zip(hdr, next(csv.reader([line.decode("utf-8", "replace")]))))                a, b = (rec.get(i_c) or "").strip(), (rec.get(o_c) or "").strip()                if a and b: rows.append((a, b))                if len(rows) >= cap: break        return rows    CODE_FENCE = re.compile(r"```")    # Translated/curated data contains the ORIGINAL model refusing. Training on those    # teaches the uncensored model to refuse - the opposite of the goal.    REFUSAL = re.compile(        r"\b(i'?m sorry|i am sorry|i apolog|sorry, but|"        r"i (?:can'?t|cannot|can not|won'?t|will not)\b|"        r"i'?m (?:not able|unable|afraid)\b|i am (?:not able|unable)\b|"        r"as an ai\b|as a language model\b|i must (?:decline|refuse)\b)\b", re.I)    MARATHI = re.compile(r"(?<![\u0900-\u097F])(आहे|आणि|मला|तुम्ही|काय|मध्ये|साठी|"                         r"करण्यासाठी|छे|पण|नाही)(?![\u0900-\u097F])")    FOREIGN = {k: re.compile(v) for k, v in {        "Bengali": r"[\u0980-\u09FF]", "Gujarati": r"[\u0A00-\u0A7F]",        "Tamil": r"[\u0A80-\u0AFF]",   "Telugu": r"[\u0B00-\u0B7F]",        "Kannada": r"[\u0B80-\u0BFF]", "Malayalam": r"[\u0D00-\u0D7F]",        "Arabic": r"[\u0600-\u06FF]"}.items()}    DEV = re.compile(r"[\u0900-\u097F]")    EN_ONLY = re.compile(r"\b(the|and|is|are|of|to|with|for|that|this|you|your|"                         r"from|have|will|can|not)\b", re.I)    # 620 chars * 1.45 tok/char + ~15 template ~= 915 tokens, inside MAX_SEQ_LENGTH=1024    MAX_CHARS = 620    MIN_WORDS = 12          # <- the check that would have caught the 8-word failure    def clean(u, a, allow_dev, min_words=MIN_WORDS):        if not u or not a or CODE_FENCE.search(u) or CODE_FENCE.search(a):            return None        if REFUSAL.search(u) or REFUSAL.search(a) or MARATHI.search(u) or MARATHI.search(a):            return None        for rx in FOREIGN.values():            if rx.search(u) or rx.search(a): return None        if allow_dev is None and bool(DEV.search(u)) != bool(DEV.search(a)):            return None                                   # mixed script: caller decides        if allow_dev is not None and bool(DEV.search(u) or DEV.search(a)) != allow_dev:            return None        if len(u) + len(a) > MAX_CHARS: return None        if len(a.split()) < min_words: return None        # terse/fragmentary answer        if len(u.split()) < 3: return None        return u, a    random.seed(3407)    out = {"hinglish": [], "hindi": [], "english": []}    prov = {}    # ---- s1: Sujalvc, AAG_anchored only --------------------------------------    # The 379 GeminiFlash rows are skipped: their replies leak English    # ("SSY is highly targeted debt scheme for girl child education").    try:        raw = grab("https://huggingface.co/datasets/Sujalvc/hinglish-instruct-dataset/"                   "resolve/main/hinglish_chat_chunk.jsonl",                   "/kaggle/working/data_cache/sujalvc.jsonl")        n = 0        for line in open(raw, encoding="utf-8"):            try: r = json.loads(line)            except Exception: continue            if r.get("source") != "AAG_anchored": continue            c = clean(r.get("instruction", ""), r.get("output", ""), allow_dev=False)            if c:                out["hinglish"].append({"instruction": c[0], "response": c[1],                                        "src": "sujalvc", "category": "AAG"})                n += 1        prov["sujalvc"] = n    except Exception as e:        print(f"  sujalvc failed: {type(e).__name__}: {e}")    # ---- s1: real human Hinglish prompts from chatbot arena -------------------    try:        rows = stream_csv("https://huggingface.co/datasets/one-thing/"                          "chatbot_arena_conversations_hinglish/resolve/main/data.csv",                          "/kaggle/working/data_cache/arena.csv",                          "user_hinglish", "assistant_hinglish", 4000)        n = 0        for u, a in rows:            c = clean(u, a, allow_dev=False, min_words=6)            if c: out["hinglish"].append({"instruction": c[0], "response": c[1],                                          "src": "arena", "category": "arena"}); n += 1        prov["arena"] = n    except Exception as e:        print(f"  arena failed: {type(e).__name__}: {e}")    # ---- s1: natural everyday register ---------------------------------------    try:        rows = stream_csv("https://huggingface.co/datasets/Abhishekcr448/"                          "Hinglish-Everyday-Conversations-1M/resolve/main/"                          "hinglish_conversations.csv",                          "/kaggle/working/data_cache/casual.csv", "input", "output", 3000)        n = 0        for u, a in rows:            c = clean(u, a, allow_dev=False, min_words=3)            if c: out["hinglish"].append({"instruction": c[0], "response": c[1],                                          "src": "casual", "category": "casual"}); n += 1        prov["casual"] = n    except Exception as e:        print(f"  casual failed: {type(e).__name__}: {e}")    # ---- s2: smangrul Devanagari + zicsx filler -------------------------------    try:        raw = grab("https://huggingface.co/datasets/smangrul/hindi_instruct_v1/"                   "resolve/main/data/train-00000-of-00001.parquet",                   "/kaggle/working/data_cache/smangrul.parquet")        import pandas as pd        df = pd.read_parquet(raw)        n = 0        for msgs, cat in zip(df["messages"], df["category"]):            try:                u = next(m["content"] for m in msgs if m["role"] == "user")                a = next(m["content"] for m in msgs if m["role"] == "assistant")            except (StopIteration, TypeError): continue            if not (DEV.search(u) and DEV.search(a)): continue      # Devanagari only            c = clean(u, a, allow_dev=True, min_words=10)            if c: out["hindi"].append({"instruction": c[0], "response": c[1],                                       "src": "smangrul", "category": str(cat)}); n += 1        prov["smangrul_deva"] = n    except Exception as e:        print(f"  smangrul failed: {type(e).__name__}: {e}")    try:   # Devanagari filler - 512k pool, clean, 0.00% Marathi        raw = grab("https://huggingface.co/datasets/zicsx/indic-align-hindi/resolve/main/"                   "data/train-00000-of-00026-a3979045df3b161a.parquet",                   "/kaggle/working/data_cache/zicsx0.parquet")        import pandas as pd        df = pd.read_parquet(raw)        n = 0        for arr in df["interactions"]:            try:                t = list(arr)[0]; u, a = str(t[0]), str(t[1])            except Exception: continue            c = clean(u, a, allow_dev=True, min_words=12)            if c: out["hindi"].append({"instruction": c[0], "response": c[1],                                       "src": "zicsx", "category": "zicsx"}); n += 1            if len(out["hindi"]) >= HINDI_TARGET * 2: break        prov["zicsx"] = n    except Exception as e:        print(f"  zicsx failed: {type(e).__name__}: {e}")    # ---- COMI-LINGUA: 3-expert-annotated, CC-BY-4.0, EMNLP-2025 ---------------    # Each MT row is triple-parallel: Sentences (code-mixed source),    # Annotator_1_RH_translation (romanized Hindi), Annotator_1_DH_translation    # (Devanagari Hindi). This is the ONLY human-expert-annotated Hindi source in    # the whole search - everything else is machine-translated or LLM-synthesised.    # Register is news/journalism at ~20 words a sentence, so it supplements the    # conversational sources rather than replacing them.    try:        import csv as _csv        p = grab("https://huggingface.co/datasets/LingoIITGN/COMI-LINGUA/resolve/main/MT_train.csv",                 "/kaggle/working/data_cache/comi_MT.csv")        nr = nd = 0        with open(p, encoding="utf-8", errors="replace") as f:            for r in _csv.DictReader(f):                src_s = (r.get("Sentences") or "").strip()                rh = (r.get("Annotator_1_RH_translation") or "").strip()                dh = (r.get("Annotator_1_DH_translation") or "").strip()                # s1: code-mixed Devanagari prompt -> romanized Hindi reply                if src_s and rh and nr < HINGLISH_TARGET:                    q = clean(src_s, rh, allow_dev=False, min_words=12)                    if q:                        out["hinglish"].append({"instruction": q[0], "response": q[1],                                                "src": "comi_rh", "category": "expert"}); nr += 1                # s2: same source -> Devanagari Hindi reply                if src_s and dh and nd < HINDI_TARGET:                    q = clean(src_s, dh, allow_dev=True, min_words=12)                    if q:                        out["hindi"].append({"instruction": q[0], "response": q[1],                                              "src": "comi_dh", "category": "expert"}); nd += 1        prov["comi_rh"] = nr        prov["comi_dh"] = nd    except Exception as e:        print(f"  comi-lingua failed: {type(e).__name__}: {e}")    # ---- s3: English anchor ---------------------------------------------------    try:        df = pd.read_parquet("/kaggle/working/data_cache/smangrul.parquet")        n = 0        for msgs, cat in zip(df["messages"], df["category"]):            try:                u = next(m["content"] for m in msgs if m["role"] == "user")                a = next(m["content"] for m in msgs if m["role"] == "assistant")            except (StopIteration, TypeError): continue            if not EN_ONLY.search(u) and not EN_ONLY.search(a): continue   # English present            if DEV.search(u) and DEV.search(a): continue                  # not pure Devanagari            c = clean(u, a, allow_dev=None, min_words=10)            if c: out["english"].append({"instruction": c[0], "response": c[1],                                         "src": "smangrul", "category": str(cat)}); n += 1        prov["smangrul_eng"] = n    except Exception as e:        print(f"  english failed: {type(e).__name__}: {e}")    # ---- mix, write, and REPORT THE THING THAT MATTERS -----------------------    def dump(name, data, n_target):        random.shuffle(data)        sel = data[:n_target]        p = f"/kaggle/working/{name}"        with open(p, "w", encoding="utf-8") as f:            for r in sel:                f.write(json.dumps(r, ensure_ascii=False) + "\n")        return sel    print("\n  source            rows kept")    for k, v in prov.items():        print(f"  {k:17s} {v:>8,}")    targets = {"s1_hinglish": ("hinglish", HINGLISH_TARGET),               "s2_hindi": ("hindi", HINDI_TARGET),               "s3_english": ("english", ENGLISH_TARGET)}    print("\n  session        rows   median-reply-words  sources")    for sess, (b, t) in targets.items():        sel = dump(f"train_{sess}.jsonl", out[b], t)        mw = sorted(len(r["response"].split()) for r in sel)        med = mw[len(mw)//2] if mw else 0        srcs = {}        for r in sel: srcs[r["src"]] = srcs.get(r["src"], 0) + 1        print(f"  {sess:13s} {len(sel):>6,}   {med:>10}          {srcs}")        if med < 12:            print(f"  !! {sess} median reply is {med} words - that is the terse-fragment "                  f"failure mode, not chat. Check the sources above.")    # script + contamination audit    print("\n--- audit ---")    for b, data in out.items():        if not data: continue        dev = sum(1 for r in data if DEV.search(r["instruction"] + r["response"]))        mar = sum(1 for r in data if MARATHI.search(r["instruction"] + r["response"]))        eng = sum(1 for r in data if len(EN_ONLY.findall(r["response"])) >= 6)        print(f"  {b:9s} n={len(data):>6,}  devanagari={dev:>6,}  marathi={mar}  english_reply={eng}")

In [ ]:
import os, glob, json, urllib.request# Data source, in priority order:#   1. a Kaggle Dataset attached to this notebook (works if the repo is private)#   2. the public GitHub repo - no Kaggle Dataset needed at allGH_RAW = "https://raw.githubusercontent.com/SujitRoy/hindi-finetune/main"# 0. built in the previous cell, 1. an attached Kaggle Dataset, 2. the public repoCANDIDATES = ([f"/kaggle/working/{DATA_FILE}"]              + [f"/kaggle/input/hinglish-sft/{DATA_FILE}"]              + glob.glob(f"/kaggle/input/*/{DATA_FILE}"))DATA_PATH = next((q for q in CANDIDATES if os.path.exists(q)), None)if DATA_PATH is None:    url = f"{GH_RAW}/{DATA_FILE}"    print(f"not attached as a Kaggle Dataset - fetching {url}")    payload = None    try:        with urllib.request.urlopen(url, timeout=180) as r:            payload = r.read()    except Exception as e:        attached = sorted(os.path.basename(d) for d in glob.glob("/kaggle/input/*"))        raise FileNotFoundError(            f"{DATA_FILE} not found. DOWNLOAD failed: {type(e).__name__}: {e}\n"            f"  datasets attached : {attached or '(none)'}\n"            f"  Fix: either attach a Kaggle Dataset, or make the GitHub repo public so\n"            f"       {url} resolves."        )    # download succeeded - now write it, and never blame the network if this fails    DATA_PATH = f"/kaggle/working/{DATA_FILE}"    try:        os.makedirs(os.path.dirname(DATA_PATH), exist_ok=True)        with open(DATA_PATH, "wb") as f:            f.write(payload)    except Exception as e:        raise RuntimeError(            f"Downloaded {len(payload):,} bytes but could not write {DATA_PATH}: "            f"{type(e).__name__}: {e}"        )    print(f"  fetched {len(payload):,} bytes -> {DATA_PATH}")rows = [json.loads(l) for l in open(DATA_PATH, encoding="utf-8")]srcs = {s: sum(1 for r in rows if r["src"] == s) for s in {r["src"] for r in rows}}print(len(rows), "rows |", srcs)dev = sum(1 for r in rows if any("\u0900" <= ch <= "\u097F" for ch in r["instruction"] + r["response"]))print(f"{dev} rows contain Devanagari")

In [ ]:
import torchMAX_SEQ_LENGTH = 1024      # p90 fits; Hindi rows are ~3x tokens/char so watch the tailDTYPE = torch.float16      # T4 is sm_75, no bf16os.environ["UNSLOTH_RETURN_LOGITS"] = "1"os.environ["UNSLOTH_VLLM_STANDBY"] = "1"   # -30% VRAM (from the official GRPO notebook)print("GPU:", torch.cuda.get_device_name(0), "| dtype:", DTYPE)

## 2. Model + LoRALFM target modules: `q,k,v,out,in_proj,w1,w2,w3` (`in_proj`/`w1,w2,w3` are its MLP).`embed_tokens`/`lm_head` are **not** trained - the tokenizer is unchanged, so there is nonew vocabulary to learn; Devanagari rides on the existing byte-level tokens.

## 2a. Build a local, self-contained tokenizerTwo upstream problems, both fixed here rather than worked around:1. `zaakirio/LFM2.5-1.2B-Instruct-Uncensored` declares `tokenizer_class: TokenizersBackend`,   a **transformers 5.x** class. Kaggle ships 4.57, so it cannot even be resolved.2. **Neither** LFM repo embeds the chat template in `tokenizer_config.json`, so   transformers has to fetch it from the Hub - and that lookup 404s on   `additional_chat_templates`, which is where a transformers 5.x expects it.So we build a local tokenizer directory: correct 4.x class, chat template embeddedinline, and a sane `model_max_length` (upstream ships `float('inf')` serialised as aninteger). After this cell, nothing touches the Hub for the tokenizer and nothing dependson transformers 5.x.Verified: vocab (64,400) and merges (63,683) are byte-identical to the abliterated repo,and this renders the same 29-token 2-turn string that `verify_format.py` validated.

In [ ]:
import json, os, urllib.requestTOKENIZER_REPO = "LiquidAI/LFM2.5-1.2B-Instruct"TOK_DIR = "/kaggle/working/tok"os.makedirs(TOK_DIR, exist_ok=True)FILES = ["tokenizer.json", "tokenizer_config.json",         "chat_template.jinja", "special_tokens_map.json"]for fn in FILES:    dst = os.path.join(TOK_DIR, fn)    if not os.path.exists(dst):        url = f"https://huggingface.co/{TOKENIZER_REPO}/resolve/main/{fn}"        with urllib.request.urlopen(url, timeout=180) as r:            open(dst, "wb").write(r.read())    print(f"  {fn}")# Embed the template and pin a real max length.cfg_path = os.path.join(TOK_DIR, "tokenizer_config.json")cfg = json.load(open(cfg_path))cfg["chat_template"] = open(os.path.join(TOK_DIR, "chat_template.jinja")).read()cfg["model_max_length"] = MAX_SEQ_LENGTH          # upstream ships float('inf') as an intcfg["tokenizer_class"] = "PreTrainedTokenizerFast" # 4.x-compatible; upstream 5.xjson.dump(cfg, open(cfg_path, "w"), ensure_ascii=False)# Prove it before the model load depends on itfrom transformers import AutoTokenizer_chk = AutoTokenizer.from_pretrained(TOK_DIR)_probe = _chk.apply_chat_template(    [{"role": "user", "content": "kya haal hai?"},     {"role": "assistant", "content": " sab badhiya"}],    tokenize=False, add_generation_prompt=False)print("tokenizer ready |", len(_chk.get_vocab()), "tokens | max_len", _chk.model_max_length)print("probe:", repr(_probe))

In [ ]:
# Weights from the abliterated repo; tokenizer from the local dir built in# cell 2a, which is self-contained and 4.x-compatible.model, tokenizer = FastLanguageModel.from_pretrained(    model_name = BASE_REPO, tokenizer_name = TOK_DIR,    max_seq_length = MAX_SEQ_LENGTH,    dtype = DTYPE, load_in_4bit = False, max_lora_rank = 32,)print("weights  :", BASE_REPO)print("tokenizer:", TOK_DIR)# Guard BEFORE attaching LoRA: named_modules() returns dotted paths like# "model.layers.0.self_attn.q_proj", so a bare "q_proj" is never an exact member.# A wrong target name attaches nothing and trains happily on zero parameters.TARGET_MODULES = ["q_proj","k_proj","v_proj","out_proj","in_proj","w1","w2","w3"]_names = [n for n, _ in model.named_modules()]_missing = [m for m in TARGET_MODULES            if not any(n == m or n.endswith("." + m) for n in _names)]if _missing:    raise RuntimeError(f"Target modules not present in this model: {_missing}")print("target modules verified:", TARGET_MODULES)model = FastLanguageModel.get_peft_model(    model, r = 32, lora_alpha = 32, lora_dropout = 0, bias = "none",    target_modules = TARGET_MODULES,    use_gradient_checkpointing = "unsloth",   # keep ON: s2 sequences are 9x longer random_state = 3407, use_rslora = True,)n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)n_all   = sum(p.numel() for p in model.parameters())print(f"trainable {n_train/1e6:.1f}M / {n_all/1e9:.2f}B ({100*n_train/n_all:.2f}%)")if n_train == 0:    raise RuntimeError("LoRA attached to zero parameters - check TARGET_MODULES")

## 3. Chat template`apply_chat_template` **already adds `<|startoftext|>`** and terminates with`<|im_end|>` - do not append an EOS yourself, and do not strip BOS twice.`train_on_responses_only` masks the user turn, so loss lands on the reply only.

In [ ]:
def to_text(batch):    convs = [[{"role":"user","content":i},{"role":"assistant","content":r}]             for i, r in zip(batch["instruction"], batch["response"])]    return {"text": [tokenizer.apply_chat_template(c, tokenize=False,                    add_generation_prompt=False) for c in convs]}from datasets import Datasetds = Dataset.from_list(rows).map(to_text, batched=True, remove_columns=list(rows[0].keys()))split = ds.train_test_split(test_size=0.02, seed=3407)print("train", len(split["train"]), "eval", len(split["test"]))print(repr(split["train"][0]["text"][:180]))

In [ ]:
from unsloth import UnslothTrainer, UnslothTrainingArgumentsargs = UnslothTrainingArguments(    per_device_train_batch_size = 8,    gradient_accumulation_steps = 1,    max_steps = MAX_STEPS,    warmup_steps = 30,    learning_rate = LEARNING_RATE,        # steps down per session: {1:2e-4, 2:1.5e-4, 3:5e-5}    lr_scheduler_type = "linear",    optim = "adamw_8bit",    weight_decay = 0.01,    eval_strategy = "steps", eval_steps = 150,    seed = 3407,    output_dir = "hf-out",    save_strategy = "no",               # the TimeCheckpoint callback owns saving    report_to = [],)trainer = UnslothTrainer(    model = model, tokenizer = tokenizer,    train_dataset = split["train"], eval_dataset = split["test"],    dataset_text_field = "text", max_seq_length = MAX_SEQ_LENGTH, args = args,)trainer.add_callback(TimeCheckpoint(model, OUT_REPO, every_s=CHECKPOINT_EVERY_S))print("trainer ready - checkpoints push every", CHECKPOINT_EVERY_S, "s")

In [ ]:
stats = trainer.train()print(stats.metrics)# Rising eval loss = the LoRA is memorising this session's slice, not learning a# language shift. Session 2/3 continue from whatever we push, so catching it here# is much cheaper than discovering it as a broken model three sessions later.evals = [(h["step"], round(h["eval_loss"], 4)) for h in trainer.state.log_history if "eval_loss" in h]if evals:    print("eval_loss by step:", evals)    if len(evals) >= 2 and evals[-1][1] > evals[0][1]:        print("WARNING: eval loss rose over training - overfitting. "              "Lower MAX_STEPS or lora_alpha before running the next session.")    else:        print("eval loss stable or falling - OK to chain into the next session.")# Unsloth resets generation_config when it switches modes, leaving max_length baked in.# That default rides along into the merged model and then into the GGUF, so clear it# BEFORE save_pretrained_gguf - afterwards it is too late.model.generation_config.max_length = Nonemodel.generation_config.max_new_tokens = Noneprint("generation_config cleared for export")

## 4. Merge -> GGUF -> Hub`save_pretrained_gguf` runs llama.cpp's converter in-notebook. Push the final merged modelso the next session - and your server - can pull it.Even if this cell never runs, the `ckpt-*` adapters pushed every 5 min are on the Hub.

### LicenseThe base model is under **Liquid Open License v1.0** (`lfm1.0`), which has 16redistribution clauses - the licence text must travel with the weights, not just themodel card. Pushed as `LICENSE` alongside the model, along with a model card recordingthe data this was trained on.

In [ ]:
from huggingface_hub import hf_hub_download, upload_fileLICENSE_SRC = "https://huggingface.co/LiquidAI/LFM2.5-1.2B-Instruct/resolve/main/LICENSE"with urllib.request.urlopen(LICENSE_SRC, timeout=180) as r:    _lic = r.read()os.makedirs("hf-out", exist_ok=True)with open("hf-out/LICENSE", "wb") as f:    f.write(_lic)upload_file(path_or_fileobj="hf-out/LICENSE", path_in_repo="LICENSE", repo_id=OUT_REPO)print(f"LICENSE pushed ({len(_lic):,} bytes) - lfm1.0 travels with the weights")

In [ ]:
model.save_pretrained_gguf("hf-out", tokenizer, quantization_method = "q4_k_m")model.push_to_hub_gguf(OUT_REPO, tokenizer, quantization_method = "q4_k_m")model.push_to_hub(OUT_REPO, tokenizer)      # session 2/3 loads thisprint("done ->", f"https://huggingface.co/{OUT_REPO}")

## 4b. Optional: iMatrix-quantized GGUF (higher quality, same size)`save_pretrained_gguf("q4_k_m")` is a **plain** quantization - it picks outliers bytensor statistics alone. iMatrix first runs the model over a calibration corpus to learnwhich *activations* actually matter, then quantizes with that importance weighting. Same`q4_k_m` bit depth, same file size, measurably less damage.This matters here specifically: the whole reason for finetuning is that the 1.2B scoresonly 5/8 on the accuracy suite, so we cannot afford to lose anything to quantization.Cost: ~5-10 min of extra session time. **Skip it if you are near the 30-minute mark** -the plain GGUF from the previous cell is a complete, working model. Pattern borrowed from`coachvitorcalvi/ava-lab-dyagnosys-finetune-v1`, which exports both and prefers imatrix.

In [ ]:
import glob, subprocessUSE_IMATRIX = True          # set False if you are short on time - plain q4_k_m is fineif not USE_IMATRIX:    print("skipped - use the q4_k_m from the previous cell")else:    # 1. calibration corpus from the actual training data (already formatted)    calib = "hf-out/calib.txt"    with open(calib, "w", encoding="utf-8") as f:        for t in split["train"]["text"][:2000]:            f.write(t + "\n\n")    print("calibration corpus:", os.path.getsize(calib), "bytes")    # 2. f16 GGUF is the input to both imatrix and quantize    model.save_pretrained_gguf("hf-out-f16", tokenizer, quantization_method = "f16")    # Unsloth appends "_gguf" to the output dir, so the file lands in hf-out-f16_gguf/    # and not hf-out-f16/. Glob both, and match on the F16 name rather than the dir.    f16 = sorted(glob.glob("hf-out-f16*/**/*.F16.gguf", recursive=True)                 or glob.glob("**/*.F16.gguf", recursive=True))    assert f16, ("no f16 GGUF found. save_pretrained_gguf writes to "                 "<dir>_gguf/, not <dir>/ - check hf-out-f16_gguf/ directly.")    f16 = f16[0]    print("f16:", f16)    # 3. Unsloth vendors a llama.cpp build; find the two binaries in it    def find_bin(name):        for root in ("/root/.unsloth/llama.cpp", "/kaggle/working/llama.cpp"):            for q in glob.glob(f"{root}/**/{name}", recursive=True):                if os.access(q, os.X_OK):                    return q        return None    imat, quant = find_bin("llama-imatrix"), find_bin("llama-quantize")    if not (imat and quant):        print("vendored build lacks the quantizers - building llama.cpp (adds ~4 min)")        d = "/kaggle/working/llama.cpp"        if not os.path.isdir(d):            subprocess.run(["git","clone","--depth","1","https://github.com/ggerganov/llama.cpp",d], check=True)        subprocess.run(["cmake","-B",f"{d}/build","-S",d,"-DLLAMA_CURL=OFF"], check=True)        subprocess.run(["cmake","--build",f"{d}/build","--config","Release","-j","-t",                        "llama-imatrix","llama-quantize"], check=True)        imat, quant = f"{d}/build/bin/llama-imatrix", f"{d}/build/bin/llama-quantize"    # 4. learn the importance matrix, then quantize with it    subprocess.run([imat,"-m",f16,"-f",calib,"-o","hf-out/imatrix.dat","--chunks","50"], check=True)    subprocess.run([quant,"--imatrix","hf-out/imatrix.dat",f16,"hf-out/gguf-imatrix","q4_k_m"], check=True)    print("iMatrix q4_k_m:", glob.glob("hf-out/gguf-imatrix*"))    # push whichever one you keep - the imatrix one if it exists    # upload the iMatrix artifact under its own name so it cannot overwrite the    # plain q4_k_m that cell 18 already pushed    from huggingface_hub import hf_hub_upload    _im = sorted(glob.glob("hf-out/gguf-imatrix*"))    if _im:        hf_hub_upload(OUT_REPO, _im[0], path_in_repo=os.path.basename(_im[0]))        print("uploaded iMatrix GGUF:", os.path.basename(_im[0]))    print("plain q4_k_m from the previous cell is still the primary artifact")

## 5. Recovering after a resetThe session died but you have checkpoints. Skip straight to the merge with the newest one:```pip install -q unslothmodel, tok = FastLanguageModel.from_pretrained(    model_name="YOUR_HF_USERNAME/lfm25-1.2b-bilingual-s1",    adapter_path="ckpt-s00600", ...)      # newest ckpt-* from the Hub```Then run the merge cell. You lose at most `CHECKPOINT_EVERY_S` of progress.